In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

from feature_engineering import create_engineered_features

PROJECT_ROOT = Path.cwd().parent
SPLIT_DIR = PROJECT_ROOT / "fine_tuning" / "data_splits"
ARTIFACT_DIR = PROJECT_ROOT / "fine_tuning" / "artifacts"

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")

reference_df = pd.concat([train_df, val_df], ignore_index=True)

with open(ARTIFACT_DIR / "feature_selection_config.json") as f:
    config = json.load(f)

reference_features = config["full_features"]

X_reference = create_engineered_features(
    reference_df.drop(columns=["TARGET", "SK_ID_CURR"])
)[reference_features]

y_reference = reference_df["TARGET"]

print("Reference rows:", len(X_reference))
print("Reference features:", len(X_reference.columns))
print("Default rate:", round(y_reference.mean(), 4))
print("Numeric features:", len(X_reference.select_dtypes(include=np.number).columns))
print("Categorical features:", len(X_reference.select_dtypes(exclude=np.number).columns))

Reference rows: 261384
Reference features: 129
Default rate: 0.0807
Numeric features: 113
Categorical features: 16


In [2]:
reference_state = {}

for col in X_reference.columns:
    series = X_reference[col]

    if pd.api.types.is_numeric_dtype(series):
        clean = series.dropna()

        reference_state[col] = {
            "type": "numeric",
            "missing_rate": float(series.isna().mean()),
            "quantiles": {
                str(q): float(clean.quantile(q))
                for q in [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1]
            }
        }
    else:
        reference_state[col] = {
            "type": "categorical",
            "missing_rate": float(series.isna().mean()),
            "frequencies": (
                series.fillna("__MISSING__")
                .value_counts(normalize=True)
                .to_dict()
            )
        }

print("Reference features captured:", len(reference_state))
print("Numeric:", sum(v["type"] == "numeric" for v in reference_state.values()))
print("Categorical:", sum(v["type"] == "categorical" for v in reference_state.values()))

Reference features captured: 129
Numeric: 113
Categorical: 16


In [3]:
import json

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(ARTIFACT_DIR / "v2_reference_state.json", "w") as f:
    json.dump(reference_state, f, indent=2)

print("Reference state saved.")
print("Path:", ARTIFACT_DIR / "v2_reference_state.json")
print("Features saved:", len(reference_state))

Reference state saved.
Path: c:\CODING\Credit_Risk\fine_tuning\artifacts\v2_reference_state.json
Features saved: 129
